<a href="https://colab.research.google.com/github/GlobalFishingWatch/gfw-api-python-client/blob/develop/notebooks/getting-started.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Getting Started

This guide introduces you to the basics of using the [gfw-api-python-client](https://github.com/GlobalFishingWatch/gfw-api-python-client).

**Note:** See the [Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset), [Data Caveats](https://globalfishingwatch.org/our-apis/documentation#data-caveat), and [Terms of Use](https://globalfishingwatch.org/our-apis/documentation#terms-of-use) pages in the [GFW API documentation](https://globalfishingwatch.org/our-apis/documentation#introduction) for details on GFW data, API licenses, and rate limits.

## Prerequisites

Before using the `gfw-api-python-client`, ensure it is installed (see the [Getting Started](https://globalfishingwatch.github.io/gfw-api-python-client/getting-started.html) guide) and that you have obtained an API access token from the [Global Fishing Watch API portal](https://globalfishingwatch.org/our-apis/tokens).

## Installation

The `gfw-api-python-client` can be easily installed using pip:

In [1]:
# %pip install gfw-api-python-client

## Usage

Import and use `gfw-api-python-client` in your Python codes

In [2]:
import os
import datetime
import pandas as pd
import gfwapiclient as gfw

In [3]:
try:
    from google.colab import userdata

    access_token = userdata.get("GFW_API_ACCESS_TOKEN")
except Exception as exc:
    access_token = os.environ.get("GFW_API_ACCESS_TOKEN")

access_token = access_token or "<PASTE_YOUR_GFW_API_ACCESS_TOKEN_HERE>"

In [4]:
gfw_client = gfw.Client(
    access_token=access_token,
)

## Making API Requests

### Searching for a Vessel by MMSI

In [5]:
vessel_search_result = await gfw_client.vessels.search_vessels(
    query="412331038",
)

In [6]:
vessel_search_df = vessel_search_result.df()

In [7]:
vessel_search_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 8 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   dataset                         5 non-null      object
 1   registry_info_total_records     5 non-null      int64 
 2   registry_info                   5 non-null      object
 3   registry_owners                 5 non-null      object
 4   registry_public_authorizations  5 non-null      object
 5   combined_sources_info           5 non-null      object
 6   self_reported_info              5 non-null      object
 7   matchCriteria                   5 non-null      object
dtypes: int64(1), object(7)
memory usage: 452.0+ bytes


In [8]:
vessel_search_df.head()

,dataset,registry_info_total_records,registry_info,registry_owners,registry_public_authorizations,combined_sources_info,self_reported_info,matchCriteria
0,public-global-vessel-identity:v3.0,0,[],[],[],[{'vessel_id': '91df2f8c7-74fd-5b5a-60f5-3d86f...,[{'id': '91df2f8c7-74fd-5b5a-60f5-3d86f9c51ff2...,[{'reference': '91df2f8c7-74fd-5b5a-60f5-3d86f...
1,public-global-vessel-identity:v3.0,1,"[{'id': '4ef90bea19300c6a23f6ce627a80238b', 's...","[{'name': 'ZHOUSHAN SHUNHANG OCEAN FISHERIES',...","[{'date_from': 2017-01-04 00:00:00+00:00, 'dat...",[{'vessel_id': '755a48dd4-4bee-4bcf-7b5f-9baea...,[{'id': '755a48dd4-4bee-4bcf-7b5f-9baea058fc7b...,[{'reference': '755a48dd4-4bee-4bcf-7b5f-9baea...
2,public-global-vessel-identity:v3.0,0,[],[],[],[{'vessel_id': 'a5ef97d59-9f40-3bdc-e247-daf9c...,[{'id': 'a5ef97d59-9f40-3bdc-e247-daf9c892ceff...,[{'reference': 'a5ef97d59-9f40-3bdc-e247-daf9c...
3,public-global-vessel-identity:v3.0,0,[],[],[],[{'vessel_id': '108c5510b-b6aa-9cf2-8b86-0598d...,[{'id': '108c5510b-b6aa-9cf2-8b86-0598de546741...,[{'reference': '108c5510b-b6aa-9cf2-8b86-0598d...
4,public-global-vessel-identity:v3.0,0,[],[],[],[{'vessel_id': 'd2fbd05f5-57dd-1faa-c384-67a75...,[{'id': 'd2fbd05f5-57dd-1faa-c384-67a75f27fc80...,[{'reference': 'd2fbd05f5-57dd-1faa-c384-67a75...


**Note:** It is recommended to prioritize vessels that include both `registry_info` and `self_reported_info` (AIS), as this indicates a successful match between registry data and AIS information.

In [9]:
vessel_search_ids = [
    self_reported_info.id
    for vessel_search_item in vessel_search_result.data()
    if vessel_search_item.registry_info_total_records >= 1
    for self_reported_info in vessel_search_item.self_reported_info
]

In [10]:
vessel_search_ids

['755a48dd4-4bee-4bcf-7b5f-9baea058fc7b',
 '3dad49b0b-b2e0-9347-0c4c-e39fea560f9f']

### Getting Details of Vessels Filtered by Vessel Searched IDs

In [11]:
vessels_result = await gfw_client.vessels.get_vessels_by_ids(
    ids=vessel_search_ids,
)

In [12]:
vessels_df = vessels_result.df()

In [13]:
vessels_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 7 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   dataset                         1 non-null      object
 1   registry_info_total_records     1 non-null      int64 
 2   registry_info                   1 non-null      object
 3   registry_owners                 1 non-null      object
 4   registry_public_authorizations  1 non-null      object
 5   combined_sources_info           1 non-null      object
 6   self_reported_info              1 non-null      object
dtypes: int64(1), object(6)
memory usage: 188.0+ bytes


In [14]:
vessels_df.head()

,dataset,registry_info_total_records,registry_info,registry_owners,registry_public_authorizations,combined_sources_info,self_reported_info
0,public-global-vessel-identity:v3.0,1,"[{'id': '4ef90bea19300c6a23f6ce627a80238b', 's...","[{'name': 'ZHOUSHAN SHUNHANG OCEAN FISHERIES',...","[{'date_from': 2017-01-04 00:00:00+00:00, 'dat...",[{'vessel_id': '755a48dd4-4bee-4bcf-7b5f-9baea...,[{'id': '755a48dd4-4bee-4bcf-7b5f-9baea058fc7b...


In [15]:
vessel_self_reported_infos = [
    self_reported_info
    for vessel_item in vessels_result.data()
    for self_reported_info in vessel_item.self_reported_info
]

In [16]:
vessel_ids = [
    self_reported_info.id for self_reported_info in vessel_self_reported_infos
]

In [17]:
vessel_ids

['755a48dd4-4bee-4bcf-7b5f-9baea058fc7b',
 '3dad49b0b-b2e0-9347-0c4c-e39fea560f9f']

### Getting Insights Related to Fishing Events for the Vessel Searched

In [18]:
start_datetime = min(
    [
        self_reported_info.transmission_date_from
        for self_reported_info in vessel_self_reported_infos
    ]
)
start_date = start_datetime.date()

**Important:** `start_date` must be on or after `January 1, 2020`

In [19]:
start_date = max(start_date, datetime.date.fromisoformat("2020-01-01"))

In [20]:
end_datetime = max(
    [
        self_reported_info.transmission_date_to
        for self_reported_info in vessel_self_reported_infos
    ]
)
end_date = end_datetime.date()

In [21]:
start_date, end_date

(datetime.date(2020, 1, 1), datetime.date(2026, 1, 4))

In [22]:
dataset_id = "public-global-vessel-identity:latest"
dataset_ids_vessel_ids = [
    {"dataset_id": dataset_id, "vessel_id": vessel_id} for vessel_id in vessel_ids
]

In [23]:
insights_result = await gfw_client.insights.get_vessel_insights(
    includes=["FISHING"],
    start_date=start_date,
    end_date=end_date,
    vessels=dataset_ids_vessel_ids,
)

In [24]:
insights_df = insights_result.df()

In [25]:
insights_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 6 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   period                       1 non-null      object
 1   vessel_ids_without_identity  0 non-null      object
 2   gap                          0 non-null      object
 3   coverage                     0 non-null      object
 4   apparent_fishing             1 non-null      object
 5   vessel_identity              0 non-null      object
dtypes: object(6)
memory usage: 180.0+ bytes


In [26]:
insights_df.head()

,period,vessel_ids_without_identity,gap,coverage,apparent_fishing,vessel_identity
0,"{'start_date': 2020-01-01, 'end_date': 2026-01...",None,None,None,{'datasets': ['public-global-fishing-events:v3...,None


In [27]:
insights_data = insights_result.data()

In [28]:
dict(insights_data.apparent_fishing.period_selected_counters)

{'events': 398,
 'events_gap_off': None,
 'events_in_rfmo_without_known_authorization': 144,
 'events_in_no_take_mpas': 0}

### Getting Fishing Events for the Vessels Searched

In [29]:
events_result = await gfw_client.events.get_all_events(
    datasets=["public-global-fishing-events:latest"],
    start_date=start_date,
    end_date=end_date,
    vessels=vessel_ids,
)

In [30]:
events_df = events_result.df()

In [31]:
events_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   start         398 non-null    datetime64[ns, UTC]
 1   end           398 non-null    datetime64[ns, UTC]
 2   id            398 non-null    object             
 3   type          398 non-null    object             
 4   position      398 non-null    object             
 5   regions       398 non-null    object             
 6   bounding_box  398 non-null    object             
 7   distances     398 non-null    object             
 8   vessel        398 non-null    object             
 9   encounter     0 non-null      object             
 10  fishing       398 non-null    object             
 11  gap           0 non-null      object             
 12  loitering     0 non-null      object             
 13  port_visit    0 non-null      object             
dtypes: datetim

In [32]:
events_df.head()

,start,end,id,type,position,regions,bounding_box,distances,vessel,encounter,fishing,gap,loitering,port_visit
0,2025-12-21 22:54:19+00:00,2025-12-22 01:59:26+00:00,20722d55b7a106978e2b55e65f7affa6,fishing,"{'lat': -45.734, 'lon': -60.6275}","{'mpa': [], 'eez': [], 'rfmo': ['ICCAT', 'IWC'...","[-60.638205, -45.736243333333334, -60.622215, ...","{'start_distance_from_shore_km': 390.0, 'end_d...",{'id': '755a48dd4-4bee-4bcf-7b5f-9baea058fc7b'...,None,"{'total_distance_km': 1.553408717842079, 'aver...",None,None,None
1,2020-05-29 15:38:07+00:00,2020-05-29 19:22:03+00:00,aa6f570aee327010bd4714aa6e459d08,fishing,"{'lat': 41.4151, 'lon': 165.5954}","{'mpa': [], 'eez': [], 'rfmo': ['NPAFC', 'PICE...","[165.58234666666667, 41.404185, 165.602265, 41...","{'start_distance_from_shore_km': 1213.0, 'end_...",{'id': '3dad49b0b-b2e0-9347-0c4c-e39fea560f9f'...,None,"{'total_distance_km': 3.2539329459390007, 'ave...",None,None,None
2,2025-12-31 23:00:02+00:00,2026-01-01 07:57:31+00:00,82ab07bcb2f0881dbdc2bea176f4d0eb,fishing,"{'lat': -45.7853, 'lon': -60.6598}","{'mpa': [], 'eez': [], 'rfmo': ['ACAP', 'CCSBT...","[-60.66967666666667, -45.80526166666667, -60.6...","{'start_distance_from_shore_km': 390.0, 'end_d...",{'id': '755a48dd4-4bee-4bcf-7b5f-9baea058fc7b'...,None,"{'total_distance_km': 5.18218418253593, 'avera...",None,None,None
3,2021-03-23 22:31:06+00:00,2021-03-24 06:45:12+00:00,634e0f958283315e9850e1523e6f7a88,fishing,"{'lat': -41.9474, 'lon': -57.9515}","{'mpa': [], 'eez': [], 'rfmo': ['ICCAT', 'IWC'...","[-57.89071, -41.96338333333333, -57.9833649999...","{'start_distance_from_shore_km': 377.0, 'end_d...",{'id': '3dad49b0b-b2e0-9347-0c4c-e39fea560f9f'...,None,"{'total_distance_km': 13.20298689870354, 'aver...",None,None,None
4,2022-06-17 12:52:22+00:00,2022-06-17 18:55:39+00:00,1c4ef3bc92b7a763c5f3b04d3a1a0779,fishing,"{'lat': 42.8639, 'lon': 168.4996}","{'mpa': [], 'eez': [], 'rfmo': ['WCPFC', 'NPAF...","[168.47266833333333, 42.858135, 168.5181933333...","{'start_distance_from_shore_km': 1122.0, 'end_...",{'id': '3dad49b0b-b2e0-9347-0c4c-e39fea560f9f'...,None,"{'total_distance_km': 4.430991184550807, 'aver...",None,None,None
